# 99 - Generate Figures

Produce publication-quality plots with uniform academic style for dissertation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Academic grayscale style
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.axisbelow": True,
})

# Output directories
FIGURES_DIR = Path("..") / "figures"
DISSERTATION_DIR = FIGURES_DIR / "dissertation"
DISSERTATION_DIR.mkdir(parents=True, exist_ok=True)

try:
    %store -r df
except:
    np.random.seed(42)
    n = 40000
    algos = np.repeat(["RSA-2048", "ECDSA-P256", "Kyber-768", "Hybrid"], n // 4)
    base = {"RSA-2048": 1500, "ECDSA-P256": 150, "Kyber-768": 80, "Hybrid": 200}
    df = pd.DataFrame({
        "algorithm": algos,
        "latency_us": [int(np.random.lognormal(np.log(base[a]), 0.3)) for a in algos],
        "timestamp_utc_iso": pd.date_range("2025-01-01", periods=n, freq="10ms"),
    })
    df["timestamp"] = df["timestamp_utc_iso"]


In [ ]:
# Figure 1: Latency CDF comparison (grayscale)
if "algorithm" in df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    
    algorithms = sorted(df["algorithm"].unique())
    linestyles = ["-", "--", "-.", ":"]
    markers = ["o", "s", "^", "D"]
    
    for i, algo in enumerate(algorithms):
        data = df[df["algorithm"] == algo]["latency_us"].values
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        # Subsample for markers
        step = max(1, len(sorted_data) // 20)
        ax.plot(sorted_data, cdf, linestyle=linestyles[i % 4], color="black",
                linewidth=1.5, label=algo)
        ax.scatter(sorted_data[::step], cdf[::step], marker=markers[i % 4],
                   s=20, color="black", facecolors="none")
    
    ax.set_xlabel("Latency (μs)")
    ax.set_ylabel("Cumulative Probability")
    ax.set_title("Cryptographic Operation Latency CDF")
    ax.legend(loc="lower right")
    
    plt.tight_layout()
    fig.savefig(DISSERTATION_DIR / "fig_latency_cdf.pdf", bbox_inches="tight")
    fig.savefig(DISSERTATION_DIR / "fig_latency_cdf.png", bbox_inches="tight")
    print(f"Saved: {DISSERTATION_DIR / 'fig_latency_cdf.pdf'}")
    plt.show()


In [ ]:
# Figure 2: Latency bar comparison
if "algorithm" in df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    
    means = df.groupby("algorithm")["latency_us"].mean()
    stds = df.groupby("algorithm")["latency_us"].std()
    p99s = df.groupby("algorithm")["latency_us"].quantile(0.99)
    
    x = np.arange(len(means))
    width = 0.35
    
    ax.bar(x - width/2, means.values, width, yerr=stds.values, capsize=4,
           label="Mean ± Std", color="gray", edgecolor="black")
    ax.bar(x + width/2, p99s.values, width, label="p99",
           color="white", edgecolor="black", hatch="//")
    
    ax.set_xticks(x)
    ax.set_xticklabels(means.index, rotation=30, ha="right")
    ax.set_ylabel("Latency (μs)")
    ax.set_title("Algorithm Latency Comparison")
    ax.legend()
    
    plt.tight_layout()
    fig.savefig(DISSERTATION_DIR / "fig_latency_comparison.pdf", bbox_inches="tight")
    fig.savefig(DISSERTATION_DIR / "fig_latency_comparison.png", bbox_inches="tight")
    print(f"Saved: {DISSERTATION_DIR / 'fig_latency_comparison.pdf'}")
    plt.show()


In [ ]:
print(f"\nAll figures saved to: {DISSERTATION_DIR.resolve()}")
print("Files:")
for f in sorted(DISSERTATION_DIR.glob("*")):
    print(f"  {f.name}")


# 99 - Generate Figures

Produce publication-quality plots for dissertation/papers.

## Objectives
- Create uniform academic style figures
- Export to dissertation/figures/
- Generate PDF and PNG versions
- Ensure reproducibility


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Academic grayscale style
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 300,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

EXPERIMENT_ID = "exp_2025_0101_001"
OUTPUT_DIR = Path(f"../figures/{EXPERIMENT_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# Load data
DATA_PATH = f"../data/{EXPERIMENT_ID}/merged/merged.parquet"
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} records")

# Generate publication-quality latency CDF
fig, ax = plt.subplots(figsize=(6, 4))
latency = df["latency_us"].sort_values()
cdf = np.arange(1, len(latency) + 1) / len(latency)

ax.plot(latency, cdf, color="black", linewidth=1.5)
ax.set_xlabel("Latency (μs)")
ax.set_ylabel("Cumulative Probability")
ax.set_title("Cryptographic Operation Latency Distribution")
ax.set_xlim(0, latency.quantile(0.99))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latency_cdf.pdf", format="pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "latency_cdf.png", format="png", dpi=300, bbox_inches="tight")
print(f"Saved: {OUTPUT_DIR / 'latency_cdf.pdf'}")
plt.show()
